# KTU S5 Machine Learning Project
## Personalized Movie Recommendation System Using Machine Learning

**Group members**
- Chris Maria Shajan C — CCE24CS069
- Chandana T D — CCE24CS068
- Gouri Sunil — CCE24CS092
- Gesmi Geenson — CCE24CS089

### Project aim
Build and evaluate a movie recommendation system using the uploaded MovieLens dataset. This notebook develops:
1. **Content-based filtering** using movie genres and user-generated tags with TF-IDF and cosine similarity.
2. **Collaborative filtering** using user rating patterns and user-user cosine similarity.
3. **Evaluation** using a held-out test set and rating prediction metrics (MAE and RMSE).

> This notebook uses real ratings from the supplied dataset. It does not diagnose users or claim that recommendations are objectively correct. Recommendation quality should be evaluated on held-out data.

## Step 0 — Install dependencies

Run this cell once in your Python environment. If your notebook environment already has these packages, no installation is needed.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn

## Step 1 — Import libraries and locate the dataset

Expected folder layout:

```text
KTU_Movie_Recommendation_Project/
├── data/
│   ├── movies.csv
│   ├── ratings.csv
│   ├── tags.csv
│   └── links.csv
└── Personalized_Movie_Recommendation.ipynb
```

If you keep the notebook in this project folder, the path below should work. If your files are somewhere else, update `DATA_DIR`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.sparse import csr_matrix

RANDOM_STATE = 42
DATA_DIR = Path("data")

# Fallback for running the notebook from a different working directory.
if not (DATA_DIR / "movies.csv").exists():
    candidates = [
        Path("/mnt/data/KTU_Movie_Recommendation_Project/data"),
        Path("../data"),
        Path("movie"),
    ]
    for candidate in candidates:
        if (candidate / "movies.csv").exists():
            DATA_DIR = candidate
            break

required_files = ["movies.csv", "ratings.csv", "tags.csv", "links.csv"]
missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Could not find {missing} in {DATA_DIR.resolve()}. "
        "Place the CSV files in a folder named 'data' beside the notebook, "
        "or update DATA_DIR."
    )

print("Dataset folder:", DATA_DIR.resolve())

## Step 2 — Load the actual CSV files

The MovieLens files contain movie metadata, ratings, tags, and external IDs.

In [ ]:
movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
links = pd.read_csv(DATA_DIR / "links.csv")

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)
print("Tags shape:", tags.shape)
print("Links shape:", links.shape)

display(movies.head())
display(ratings.head())
display(tags.head())

## Step 3 — Check data quality

We inspect missing values, duplicate rows, rating ranges, and the number of unique users and movies.

In [ ]:
print("Missing values in movies:")
display(movies.isna().sum().to_frame("missing_count"))

print("Missing values in ratings:")
display(ratings.isna().sum().to_frame("missing_count"))

print("Duplicate movie IDs:", movies["movieId"].duplicated().sum())
print("Duplicate user-movie rating pairs:",
      ratings.duplicated(subset=["userId", "movieId"]).sum())
print("Rating range:", ratings["rating"].min(), "to", ratings["rating"].max())
print("Unique users:", ratings["userId"].nunique())
print("Unique rated movies:", ratings["movieId"].nunique())

## Step 4 — Exploratory data analysis (EDA)

These plots help us understand the dataset before modelling.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=ratings, x="rating", color="steelblue")
plt.title("Distribution of Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.tight_layout()
plt.show()

rating_counts = ratings.groupby("movieId").size()
plt.figure(figsize=(8, 4))
plt.hist(rating_counts, bins=40, edgecolor="black")
plt.title("Number of Ratings per Movie")
plt.xlabel("Ratings per movie")
plt.ylabel("Number of movies")
plt.tight_layout()
plt.show()

movies["genre_list"] = movies["genres"].fillna("").apply(
    lambda value: [g for g in value.split("|") if g and g != "(no genres listed)"]
)
genre_counts = movies.explode("genre_list")["genre_list"].value_counts()
plt.figure(figsize=(10, 5))
genre_counts.head(15).sort_values().plot(kind="barh")
plt.title("15 Most Common Movie Genres")
plt.xlabel("Number of movies")
plt.tight_layout()
plt.show()

display(genre_counts.head(15).to_frame("movie_count"))

## Step 5 — Build content features from genres and tags

For each movie, we combine its genres and user tags into a text description. TF-IDF converts this text into numerical features, and cosine similarity measures similarity between movies.

In [ ]:
# Aggregate tags per movie. The dataset may contain repeated tags; that is acceptable.
tags["tag"] = tags["tag"].fillna("").astype(str)
tag_text = (
    tags.groupby("movieId")["tag"]
    .apply(lambda values: " ".join(values))
    .rename("tag_text")
)

content_movies = movies[["movieId", "title", "genres"]].copy()
content_movies = content_movies.merge(tag_text, on="movieId", how="left")
content_movies["genres"] = content_movies["genres"].fillna("").str.replace("|", " ", regex=False)
content_movies["tag_text"] = content_movies["tag_text"].fillna("")
content_movies["content"] = (
    content_movies["genres"] + " " + content_movies["tag_text"]
).str.strip()

# A small fallback ensures movies without tags still have genre features.
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(content_movies["content"])

print("Content feature matrix shape:", tfidf_matrix.shape)
print("Number of TF-IDF terms:", len(tfidf.get_feature_names_out()))

## Step 6 — Content-based recommendation function

Give a movie title (or part of one). The function returns movies with similar genre/tag content. It first tries an exact case-insensitive match, then a partial title match.

In [ ]:
# Similarities are computed only for the requested movie to avoid allocating a huge dense matrix.
title_to_index = pd.Series(content_movies.index, index=content_movies["title"].str.lower())
# Some datasets can contain duplicate titles; use the first matching index.

def recommend_similar_movies(movie_title, n=10):
    query = str(movie_title).strip().lower()
    exact = content_movies.index[content_movies["title"].str.lower() == query].tolist()
    if exact:
        idx = exact[0]
    else:
        partial = content_movies.index[
            content_movies["title"].str.lower().str.contains(query, regex=False, na=False)
        ].tolist()
        if not partial:
            return pd.DataFrame(columns=["title", "genres", "similarity"])
        idx = partial[0]
        print("Using closest title match:", content_movies.loc[idx, "title"])

    scores = list(enumerate(cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()))
    scores = sorted(scores, key=lambda item: item[1], reverse=True)
    scores = [(i, score) for i, score in scores if i != idx][:n]

    result = content_movies.iloc[[i for i, _ in scores]][["title", "genres"]].copy()
    result["similarity"] = [score for _, score in scores]
    return result.reset_index(drop=True)

# Example:
recommend_similar_movies("Toy Story (1995)", n=10)

## Step 7 — Prepare a train/test split for collaborative filtering

We hold out some known ratings for evaluation. The model will build user profiles using training ratings only. This makes evaluation more meaningful than testing on the same ratings used to construct the model.

In [ ]:
train_ratings, test_ratings = train_test_split(
    ratings[["userId", "movieId", "rating"]].copy(),
    test_size=0.20,
    random_state=RANDOM_STATE
)

user_ids = sorted(ratings["userId"].unique())
movie_ids = sorted(movies["movieId"].unique())
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

train_rows = train_ratings["userId"].map(user_to_idx).to_numpy()
train_cols = train_ratings["movieId"].map(movie_to_idx).to_numpy()
train_vals = train_ratings["rating"].to_numpy(dtype=float)

user_item_train = csr_matrix(
    (train_vals, (train_rows, train_cols)),
    shape=(len(user_ids), len(movie_ids))
)

print("Training ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))
print("User-item matrix shape:", user_item_train.shape)
print("Matrix density:", round(user_item_train.nnz / (user_item_train.shape[0] * user_item_train.shape[1]), 5))

## Step 8 — User-based collaborative filtering

The method finds users with similar rating patterns using cosine similarity. To predict a user's rating for a movie, it takes a similarity-weighted average of ratings for that movie from similar users who rated it.

If no similar user has rated the movie, it falls back to the training-set mean. This is a simple, explainable baseline suitable for an academic project.

In [ ]:
# User-user similarity is calculated from the training matrix only.
user_similarity = cosine_similarity(user_item_train, dense_output=True)

global_mean = float(train_ratings["rating"].mean())
user_means = train_ratings.groupby("userId")["rating"].mean().to_dict()

def predict_rating_user_cf(user_id, movie_id, k=30):
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return global_mean

    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]

    # Users who rated this movie in the training data
    movie_col = user_item_train[:, midx].toarray().ravel()
    candidate_users = np.flatnonzero(movie_col > 0)

    # Exclude the target user from neighbours
    candidate_users = candidate_users[candidate_users != uidx]
    if len(candidate_users) == 0:
        return user_means.get(user_id, global_mean)

    sims = user_similarity[uidx, candidate_users]
    top_positions = np.argsort(sims)[::-1][:k]
    neighbour_idx = candidate_users[top_positions]
    neighbour_sims = sims[top_positions]
    neighbour_ratings = movie_col[neighbour_idx]

    # Keep only positive-similarity neighbours
    valid = neighbour_sims > 0
    if not np.any(valid):
        return user_means.get(user_id, global_mean)

    weights = neighbour_sims[valid]
    values = neighbour_ratings[valid]
    prediction = float(np.dot(weights, values) / weights.sum())

    # MovieLens ratings range from 0.5 to 5.0.
    return float(np.clip(prediction, 0.5, 5.0))

# Quick example using a row from the test set
sample = test_ratings.iloc[0]
print("User:", int(sample.userId))
print("Movie ID:", int(sample.movieId))
print("Actual rating:", float(sample.rating))
print("Predicted rating:", round(predict_rating_user_cf(int(sample.userId), int(sample.movieId)), 3))

## Step 9 — Evaluate collaborative filtering

This cell evaluates the model on a subset of held-out ratings first, so it finishes faster on a regular laptop. Increase `EVALUATION_ROWS` to evaluate more test ratings.

In [ ]:
EVALUATION_ROWS = min(2000, len(test_ratings))
evaluation_sample = test_ratings.sample(
    n=EVALUATION_ROWS,
    random_state=RANDOM_STATE
).reset_index(drop=True)

actual = evaluation_sample["rating"].to_numpy(dtype=float)
predicted = []

for row in evaluation_sample.itertuples(index=False):
    predicted.append(
        predict_rating_user_cf(int(row.userId), int(row.movieId), k=30)
    )

predicted = np.asarray(predicted)
mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))

print(f"Evaluated ratings: {len(actual)}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

plt.figure(figsize=(6, 5))
plt.scatter(actual, predicted, alpha=0.25)
plt.xlabel("Actual rating")
plt.ylabel("Predicted rating")
plt.title("Actual vs Predicted Ratings (User-based CF)")
plt.xlim(0.5, 5)
plt.ylim(0.5, 5)
plt.tight_layout()
plt.show()

## Step 10 — Generate personalized recommendations for a user

The function recommends movies that the user has not rated in the training data. It predicts ratings for candidate movies and returns the highest-scoring ones. This is a demonstration of collaborative filtering personalization.

In [ ]:
def recommend_for_user(user_id, n=10, min_ratings_for_candidate=5):
    if user_id not in user_to_idx:
        raise ValueError(f"User {user_id} is not in the ratings dataset.")

    already_rated = set(
        train_ratings.loc[train_ratings["userId"] == user_id, "movieId"].tolist()
    )

    # Consider movies that have enough ratings in the training set.
    movie_rating_counts = train_ratings.groupby("movieId").size()
    candidates = [
        mid for mid in movie_ids
        if mid not in already_rated and movie_rating_counts.get(mid, 0) >= min_ratings_for_candidate
    ]

    scored = []
    for movie_id in candidates:
        score = predict_rating_user_cf(user_id, movie_id, k=30)
        scored.append((movie_id, score))

    top = sorted(scored, key=lambda item: item[1], reverse=True)[:n]
    result = pd.DataFrame(top, columns=["movieId", "predicted_rating"])
    result = result.merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
    return result[["title", "genres", "predicted_rating"]].reset_index(drop=True)

# Example: choose an existing user ID from the dataset.
example_user_id = int(ratings["userId"].iloc[0])
print("Example user ID:", example_user_id)
recommend_for_user(example_user_id, n=10)

## Step 11 — Inspect a user's previous ratings

This helps explain recommendations during a demo or viva.

In [ ]:
def show_user_ratings(user_id, n=15):
    history = train_ratings[train_ratings["userId"] == user_id].merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )
    return history.sort_values("rating", ascending=False)[
        ["title", "genres", "rating"]
    ].head(n).reset_index(drop=True)

show_user_ratings(example_user_id)

## Step 12 — Save processed outputs (optional)

This saves a compact copy of the recommendation results for use in a report. The raw dataset files remain unchanged.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

content_example = recommend_similar_movies("Toy Story (1995)", n=10)
content_example.to_csv(OUTPUT_DIR / "content_based_example.csv", index=False)

collaborative_example = recommend_for_user(example_user_id, n=10)
collaborative_example.to_csv(OUTPUT_DIR / "collaborative_example.csv", index=False)

print("Saved example recommendations to:", OUTPUT_DIR.resolve())

## Step 13 — Results and report checklist

After running all cells, record the actual results from your run. Do not invent or copy evaluation values from another project.

Include these points in the report:
1. Dataset dimensions and rating range.
2. Missing-value and duplicate checks.
3. EDA plots and observations.
4. Content-based method: TF-IDF and cosine similarity.
5. Collaborative method: user-user cosine similarity and weighted rating prediction.
6. Test split and actual MAE/RMSE values.
7. Examples of recommendations and their limitations.
8. Future improvements: item-based CF, matrix factorization (SVD), hybrid ranking, diversity, and a Streamlit interface.

### Important limitations
- The dataset does not contain movie posters or full plot descriptions. The first version uses genres and tags for content similarity.
- A user's historical ratings are used as preference signals; the system does not know their personal identity.
- The simple user-based CF model may be slow for very large datasets. This MovieLens version is manageable for a student project, but model optimization can be added later.
- A new user with no rating history faces the cold-start problem. A practical interface can ask them to select a few favourite movies or genres first.